In [ ]:
import os, subprocess, sys, json, time, glob
os.environ['PYTHONUTF8']='1'
print(subprocess.run([sys.executable,'-m','pip','install','-q','flopscope>=0.12.3,<0.13','whestbench>=0.16.2,<0.17'],capture_output=True,text=True).stdout[-500:])
from importlib.metadata import version; print('flopscope', version('flopscope'), 'whestbench', version('whestbench'))
print('cpus', os.cpu_count()); print(subprocess.run(['free','-g'],capture_output=True,text=True).stdout)

In [ ]:
hits = glob.glob('/kaggle/input/**/x11_pipeline.py', recursive=True); print('pipeline at', hits)
INPUT = os.path.dirname(hits[0])
print(sorted(os.listdir(INPUT)))

In [ ]:
os.environ['HF_XET_HIGH_PERFORMANCE']='1'
t0=time.time()
r=subprocess.run(['whest','dataset','download','aicrowd/arc-whestbench-public-2026','--revision','v2-phase2','--split','mini'],capture_output=True,text=True)
print(r.stdout[-800:], r.stderr[-800:]); print('download s', time.time()-t0)

In [ ]:
env = dict(os.environ); env.update({"CAND": "C2", "CAND_ENV": "{\"X8_D21S\": \"0.98\", \"X2_K31_MIX\": \"0.4\", \"X8_SLOC\": \"0.2\", \"X8_SSTAR\": \"0.7\", \"V21_R_OLD\": \"416\", \"X_DEAD_TAU_B\": \"2.4\"}", "NPROC": "4", "STAGES": "eval,sum", "EVAL_NETS": "70-77,0-3;85-92,40-43;93-99,60-62"})
env.update({'INPUT': INPUT, 'WORK': '/kaggle/working', 'PYTHONWARNINGS': 'ignore', 'PYTHONUTF8': '1'})
t0 = time.time()
p = subprocess.Popen([sys.executable, INPUT + '/x11_pipeline.py'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' in line or 'warnings.warn' in line: continue
    print(line, end='', flush=True)
p.wait(); print('exit', p.returncode, 'elapsed', time.time() - t0)
print(subprocess.run(['du', '-sh', '/kaggle/working'], capture_output=True, text=True).stdout)